# DAVID-Net - Build the clip cache (Phase A)

Run this **once**, on a **CPU** session (Settings -> Accelerator -> **None**). It spends
no GPU quota.

Every clip in the campaign partition is decoded once into ~480 KB of tiled JPEG plus
int16 PCM and packed into large shards. Training then reads a clip with one `pread` and
one JPEG decode instead of forking an ffmpeg subprocess per sample -- which is what
pushed the container past its 32 GB memory cgroup and had every Stage-1 run killed
(SIGKILL, exit 137, "Canceled by backend") about five minutes in.

**How to run it**

1. Settings -> Accelerator: **None**. Internet: **On**. Add the FakeAVCeleb dataset.
2. Add your `HF_TOKEN` secret (Add-ons -> Secrets), same as the training notebook.
3. **Save Version -> Save & Run All (Commit)**. Not "Run All" in the editor: an
   interactive session dies with the browser tab.
4. It takes roughly 1-2 hours and is **resumable** - if it hits the 12 h wall or you
   stop it, commit it again and it continues from the index it already wrote.
5. When it finishes, open the version's **Output** tab -> **New Dataset**, and name it
   `davidnet-av-cache`. Make it **public** so all 10 accounts can attach it without
   sharing invitations.
6. In the training notebook: **+ Add Input -> Datasets -> `davidnet-av-cache`**. Cell 8
   finds it automatically; there is nothing to edit.

In [ ]:
# Cell 1: GPU check + install deps
import os
# --- keep this notebook's output small -------------------------------------------------
# An Interactive Kaggle session is tethered to the browser websocket, and Kaggle reaps the
# container when that connection stops heartbeating. The surest way to break it is an
# output flood: huggingface_hub upload bars and transformers' "Loading weights" emit
# thousands of carriage-return redraws per run. Silence them BEFORE anything imports them
# (subprocesses inherit os.environ, so this covers Cells 9 and 11 too).
os.environ['PYTHONUNBUFFERED'] = '1'   # stream subprocess logs line by line (no 8 KB pipe buffering)
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TQDM_DISABLE'] = '1'
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU.")
!pip install -q transformers accelerate scikit-learn jiwer datasets
import shutil; assert shutil.which('ffmpeg'), 'ffmpeg missing - decoding needs it'
!df -h /dev/shm /kaggle/working | tail -n +1

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token + start the SESSION watchdog
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

# Session-level telemetry -> HF runs/session_<id>/logs/watchdog_session.jsonl, one line per
# minute (RAM, VRAM, disk, /dev/shm, GPU util, ffmpeg count, current cell). Lives in the
# notebook kernel: if the TRAINING process dies, this keeps reporting; if THIS stops, the
# session itself was killed and its last line is the time of death.
# Name THIS account so `publish_campaign.py status` shows who holds each lease. Kaggle
# does not expose the owner in a batch session, so set it per account (any short label).
WORKER_NAME = ""          # e.g. "acct-01"; blank falls back to the container hostname
if WORKER_NAME:
    os.environ["DAVIDNET_WORKER"] = WORKER_NAME

import time as _time
SESSION_ID = _time.strftime("%Y%m%d_%H%M%S")
CURRENT_CELL = {"name": "cell3"}
def _on_pre_run(info):
    src = (info.raw_cell or "").strip().splitlines()
    CURRENT_CELL["name"] = src[0][:60] if src else "?"
get_ipython().events.register("pre_run_cell", _on_pre_run)
from src.utils.watchdog import start_session_watchdog
SESSION_WD = start_session_watchdog(SESSION_ID, local_dir="/kaggle/working",
                                    current_cell=lambda: CURRENT_CELL["name"])
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "?")
print(f"session watchdog started: runs/session_{SESSION_ID}/logs/watchdog_session.jsonl  "
      f"(run type: {RUN_TYPE})")
if RUN_TYPE != "Batch":
    print("\n" + "!" * 78)
    print("!! INTERACTIVE SESSION - this container dies with your browser tab.")
    print("!! A reap looks exactly like a silent stop: healthy RAM/VRAM/disk, no traceback,")
    print("!! no signal, both watchdogs ending in the same minute. Stage 1 is ~30 GPU-hours")
    print("!! and cannot finish here regardless.")
    print("!! For training, use:  Save Version -> Save & Run All (Commit)")
    print("!" * 78 + "\n")


In [ ]:
# Cell 3b: this notebook needs FakeAVCeleb and nothing else.
#
# Version 1 of this notebook failed after 283 s with "No space left on device". Cell 4
# auto-downloads any dataset that is not mounted, and since only FakeAVCeleb was
# attached it began fetching dfdc-10 and then a 96.5 GB corpus into a 20 GB disk. The
# campaign partition is FakeAVCeleb-only, so the other six are simply not wanted here.
DOWNLOAD_ALLOWLIST = {"fakeavceleb"}
print(f"download allowlist: {DOWNLOAD_ALLOWLIST}")

In [ ]:
# Cell 4: Discover datasets; download only the ones this notebook declared it needs
import shutil
import subprocess
from pathlib import Path

# A notebook that needs a subset sets DOWNLOAD_ALLOWLIST before this cell. Default is
# everything, which is what the training notebook wants. The cache builder sets
# {"fakeavceleb"}: told to attach that alone, this cell used to start downloading the
# other six -- one of them 96.5 GB -- into a 20 GB disk, and the session died on ENOSPC
# before a single clip was decoded.
DOWNLOAD_ALLOWLIST = globals().get("DOWNLOAD_ALLOWLIST", None)
MIN_FREE_GB_TO_DOWNLOAD = 12.0

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
DOWNLOAD_DIR = WORKING / "kaggle_datasets"
DOWNLOAD_DIR.mkdir(exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg"}

def has_media_files(d):
    for f in d.rglob("*"):
        if f.suffix.lower() in VIDEO_EXTS | AUDIO_EXTS:
            return True
    return False

DATASETS = {
    "fakeavceleb": {
        "slug": "aicontentdetections/fakeavceleb-v1-2",
        "mounts": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    },
    "dfdc-10": {
        "slug": "pranay22077/dfdc-10",
        "mounts": ["pranay22077/dfdc-10"],
    },
    "deepfaketimit": {
        "slug": "fahimaislam1812/deepfaketimit",
        "mounts": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    },
    "celeb-df-v2": {
        "slug": "reubensuju/celeb-df-v2",
        "mounts": ["reubensuju/celeb-df-v2"],
    },
    "asvpoof-2019": {
        "slug": "anishsarkar22/asvpoof-2019-dataset-la",
        "mounts": ["anishsarkar22/asvpoof-2019-dataset-la"],
    },
    "in-the-wild": {
        "slug": "abdallamohamed312/in-the-wild-audio-deepfake",
        "mounts": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    },
    "wavefake": {
        "slug": "walimuhammadahmad/fakeaudio",
        "mounts": ["walimuhammadahmad/fakeaudio", "andreadiubaldo/wavefake-test"],
    },
}

if DOWNLOAD_ALLOWLIST is None:
    DOWNLOAD_ALLOWLIST = set(DATASETS)


def find_mounted(slug_paths):
    for p in slug_paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                return candidate
    return None

def download_dataset(slug, friendly):
    dst = DOWNLOAD_DIR / friendly
    if dst.exists() and has_media_files(dst):
        print(f"  {friendly}: already downloaded")
        return dst
    free_gb = shutil.disk_usage(str(WORKING)).free / 1e9
    if free_gb < MIN_FREE_GB_TO_DOWNLOAD:
        print(f"  {friendly}: SKIPPED -- only {free_gb:.1f} GB free, need "
              f"{MIN_FREE_GB_TO_DOWNLOAD:.0f} GB of headroom")
        return None
    dst.mkdir(parents=True, exist_ok=True)
    print(f"  {friendly}: downloading from {slug} ({free_gb:.1f} GB free)...", end=" ")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", str(dst), "--unzip"],
            capture_output=True, text=True, timeout=3600
        )
        if result.returncode == 0:
            print("OK")
            return dst
        else:
            print(f"FAIL: {result.stderr[:200]}")
            return None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

datasets = {}
for friendly, info in DATASETS.items():
    mounted = find_mounted(info["mounts"])
    if mounted:
        datasets[friendly] = mounted
        print(f"  {friendly} -> {mounted} (mounted)")
        continue
    downloaded = DOWNLOAD_DIR / friendly
    if downloaded.exists() and has_media_files(downloaded):
        datasets[friendly] = downloaded
        print(f"  {friendly} -> {downloaded} (cached)")
        continue
    if friendly not in DOWNLOAD_ALLOWLIST:
        print(f"  {friendly}: not mounted, not needed by this notebook -> skipped")
        continue
    result = download_dataset(info["slug"], friendly)
    if result:
        datasets[friendly] = result

print(f"\nFound {len(datasets)}/{len(DATASETS)} datasets "
      f"({len(DOWNLOAD_ALLOWLIST)} were eligible to download).")
_missing = set(DATASETS) - set(datasets)
if _missing:
    print(f"Not present: {_missing}")
print(f"disk: {shutil.disk_usage(str(WORKING)).free / 1e9:.1f} GB free in {WORKING}")

In [ ]:
# Cell 5: Extract compressed datasets (multi-part zips, tar, etc.)
import zipfile, tarfile, shutil

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def find_first_zip_part(src):
    """Find the .001 part of a multi-part zip, anywhere in tree."""
    best = None
    best_num = 999999
    count = 0
    for f in src.rglob("*"):
        name = f.name
        # Match patterns like: foo.zip.001, foo.zip.016
        if ".zip." in name:
            parts = name.split(".zip.")
            if len(parts) == 2 and parts[1].isdigit():
                count += 1
                num = int(parts[1])
                if num < best_num:
                    best_num = num
                    best = f
    if best:
        print(f"    Found {count} zip parts, first = {best.name} (part {best_num})")
    return best, count

def extract_multipart_zip(name, first_part, dst, search_root=None):
    print(f"  {name}: extracting multi-part zip from {first_part.name}...")
    if search_root is None:
        search_root = first_part.parent
    stem = first_part.name.split(".zip.")[0]
    # Find ALL parts across all subdirs
    all_parts = sorted(search_root.rglob(f"{stem}.zip.*"),
                       key=lambda x: int(x.name.split(".zip.")[1]))
    print(f"  {name}: found {len(all_parts)} parts across subdirs")

    # Method 1: Try 7z â€” copy all parts to temp dir first (7z needs them co-located)
    try:
        import shutil, tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp = Path(tmpdir)
            for p in all_parts:
                shutil.copy2(str(p), str(tmp / p.name))
            first_tmp = tmp / first_part.name
            result = subprocess.run(
                ["7z", "x", str(first_tmp), f"-o{dst}", "-y"],
                capture_output=True, text=True, timeout=600
            )
            if result.returncode == 0:
                print(f"  {name}: OK (via 7z)")
                return True
            print(f"  {name}: 7z failed: {result.stderr[:200]}")
    except FileNotFoundError:
        print(f"  {name}: 7z not found, trying concat method...")
    except subprocess.TimeoutExpired:
        print(f"  {name}: 7z timed out")

    # Method 2: Concatenate all parts into one zip, then extract
    # Search from dataset root (not just .001 parent) since parts may be in sibling dirs
    # Method 2: Concatenate all parts into one zip, then extract
    try:
        print(f"  {name}: concatenating {len(all_parts)} parts...")
        merged = dst / f"{stem}_merged.zip"
        with open(merged, "wb") as out:
            for part in all_parts:
                with open(part, "rb") as inp:
                    while True:
                        chunk = inp.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        out.write(chunk)
        print(f"  {name}: merged to {merged.stat().st_size // 1024 // 1024}MB, extracting...")
        with zipfile.ZipFile(str(merged)) as zf:
            zf.extractall(dst)
        merged.unlink()  # remove merged zip to save space
        print(f"  {name}: OK (via concat)")
        return True
    except Exception as e:
        print(f"  {name}: FAILED: {e}")
        return False

def extract_archives(name, src, dst):
    extracted = False
    # Multi-part zip first â€” pass src as search_root so we find parts across all subdirs
    first_part, count = find_first_zip_part(src)
    if first_part:
        extracted = extract_multipart_zip(name, first_part, dst, search_root=src)
    # Regular zips
    for arch in src.rglob("*.zip"):
        if ".zip." in arch.name:
            continue
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    # Tar.gz
    for arch in src.rglob("*.tar.gz"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with tarfile.open(arch, "r:gz") as tf:
                tf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    return extracted

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already extracted")
        continue
    if has_media_files(path):
        print(f"  {name}: loose media files found")
        marker.touch()
        continue
    # Check disk space â€” skip if dataset too large for Kaggle (~20GB working)
    import shutil as _shutil
    free_gb = _shutil.disk_usage(str(DATA_DIR)).free / (1024**3)
    zip_count = sum(1 for _ in path.rglob("*.zip.*") if '.zip.' in _.name and _.name.split('.zip.')[1].isdigit())
    est_gb = zip_count * 1.0  # each part is ~1GB
    if est_gb > free_gb * 0.85:
        print(f"  {name}: SKIPPED â€” {est_gb:.0f}GB needed but only {free_gb:.1f}GB free")
        continue
    print(f"  {name}: no loose media, searching archives...")
    try:
        extract_archives(name, path, dst)
    except OSError as e:
        print(f"  {name}: FAILED (disk error): {e}")
        # Clean up partial extraction to free space
        import shutil as _shutil2
        if dst.exists():
            _shutil2.rmtree(dst, ignore_errors=True)
        continue
    if has_media_files(dst):
        print(f"  {name}: extracted media OK")
    else:
        print(f"  {name}: WARNING - no media files after extraction")
    marker.touch()

print("\nExtraction done.")

In [ ]:
# Cell 6: Build ALL manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# Campaign determinism: a mounted Kaggle dataset always resolves to its LATEST version, so
# rebuilding manifests on a different account can silently produce a DIFFERENT subject
# split -- incomparable seeds, and clips that are test here but train there. Once the
# campaign partition is published, never rebuild it; Cell 7 fetches and hash-verifies it.
from src.utils.splits_sync import published_index
SPLITS_PUBLISHED = published_index() is not None
print("campaign splits already published on HF -> skipping rebuild"
      if SPLITS_PUBLISHED else "no campaign splits yet -> building (Cell 7 publishes them)")

# === FakeAVCeleb ===
fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    # The manifest's rel_path is relative to THIS directory -> training must use the
    # same root. (Run 1 passed the mount root instead; every clip was "missing" and the
    # loader silently trained on random tensors. The loader now raises instead.)
    datasets["fakeavceleb"] = fakeav_root
    print(f"FakeAVCeleb: {fakeav_root}")
    if not SPLITS_PUBLISHED:
        !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# === All other datasets ===
CONVERTERS = [
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    extracted_root = DATA_DIR / ds_name
    if not root or not root.exists():
        if extracted_root.exists() and has_media_files(extracted_root):
            root = extracted_root
            print(f"  {ds_name}: using extracted path {root}")
    if root and root.exists():
        print(f"\n--- {ds_name} ---")
        if not SPLITS_PUBLISHED:
            !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}
    else:
        print(f"  {ds_name}: NOT FOUND")

print("\n" + "="*50)
print("ALL MANIFESTS:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Training manifests = the ONE published, hash-verified campaign partition
import json, yaml
from collections import Counter
from src.utils.splits_sync import fetch_and_verify, published_index, publish

FAKEAV_ROOT = str(datasets["fakeavceleb"])   # the mount; see the cache cell below

# NOTE: an earlier version copied FakeAVCeleb into /kaggle/working first. Do not
# bring that back. Copying 6.6 GB charged ~20 GB to the container's 32 GB cgroup --
# the bytes read from the mount, the dirty pages written to /dev/loop2, and the loop
# device's own backing cache -- leaving training 4 GB of headroom and earning an
# immediate SIGKILL (exit 137). Clips are read straight from the mount now, and only
# once, by the cache builder.

# The first worker in a campaign freezes the partition; every later worker -- on any
# account -- downloads exactly those bytes and verifies SHA-256 before touching a GPU.
# SPLITS_SHA256.json is also what the paper cites so reviewers can reproduce the split.
if published_index() is None:
    print("publishing this session's splits as the campaign partition...")
    publish(str(SPLIT_DIR), extra_provenance={"fakeav_root": FAKEAV_ROOT})

VERIFIED_SPLITS = WORKING / "splits_verified"
fetch_and_verify(str(VERIFIED_SPLITS))              # raises SystemExit on any mismatch

FAKEAV_SPLITS = VERIFIED_SPLITS / "fakeavceleb"
TRAIN_MANIFEST = FAKEAV_SPLITS / "train.jsonl"
VAL_MANIFEST = FAKEAV_SPLITS / "val.jsonl"
TEST_MANIFEST = FAKEAV_SPLITS / "test.jsonl"
for m in (TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST):
    assert m.exists(), f"missing split {m} - the published partition is incomplete"

def _summary(path):
    recs = [json.loads(l) for l in open(path) if l.strip()]
    return len(recs), dict(Counter(r["quadrant"] for r in recs)), dict(Counter(r["generator"] for r in recs))

for name, m in [("train", TRAIN_MANIFEST), ("val", VAL_MANIFEST), ("test", TEST_MANIFEST)]:
    n, quads, gens = _summary(m)
    print(f"{name:5s}: {n:6d} clips  quadrants={quads}")
    if name == "train":
        print(f"       generators={gens}")

# Hard preflight: the first train clip must exist under FAKEAV_ROOT and decode to real
# frames/audio (this is the check that would have caught run 1 at t=0).
first = json.loads(open(TRAIN_MANIFEST).readline())
probe = Path(FAKEAV_ROOT) / first["rel_path"]
assert probe.exists(), f"{probe} does not exist -> FAKEAV_ROOT is wrong"
from src.data.decode import decode_clip
d = decode_clip(str(probe), 16, 64000, 224, window="center")
print(f"preflight OK: {probe.name} video{tuple(d.video.shape)} std={d.video.std():.3f} "
      f"audio{tuple(d.audio.shape)} std={d.audio.std():.4f} has_audio={d.has_audio}")
assert d.has_audio and d.audio.std() > 1e-4 and d.video.std() > 1e-3, "decoded clip looks empty"

## Build

In [ ]:
# Build the cache. Resumable: re-committing this notebook continues where it stopped.
#
# --workers 4 matches Kaggle's 4 vCPUs. The builder recycles each worker process every
# 200 clips and drops every finished shard from page cache (POSIX_FADV_DONTNEED), so its
# memory footprint stays flat however many clips it has processed -- the discipline the
# training loop was missing.
#
# --time-budget-min stops cleanly with the index written, rather than being cut off
# mid-shard by the 12 h session wall.
CACHE_OUT = WORKING / "av_cache"
MOUNT = Path(FAKEAV_ROOT)

import shutil as _sh
print(f"source : {MOUNT}")
print(f"target : {CACHE_OUT}")
_free = _sh.disk_usage(str(WORKING)).free / 1e9
print(f"free   : {_free:.1f} GB (the full cache is about 10.5 GB)")
assert _free > 12.0, (
    f"only {_free:.1f} GB free in /kaggle/working. Something else has filled the disk -- "
    "check that no extra datasets were downloaded (Cell 3b limits that to FakeAVCeleb).")

!cd {REPO} && python -m scripts.build_clip_cache \
    --manifest {TRAIN_MANIFEST} {VAL_MANIFEST} {TEST_MANIFEST} \
    --root "{MOUNT}" \
    --out "{CACHE_OUT}" \
    --workers 4 --maxtasksperchild 200 --time-budget-min 660

## Verify

In [ ]:
# Verify before publishing: coverage per split, and one clip decoded end to end.
from src.data.clipcache import ClipCache

cache = ClipCache(CACHE_OUT)
print(f"{len(cache)} clips cached, {len(cache.failures)} undecodable")

ok = True
for name, m in [("train", TRAIN_MANIFEST), ("val", VAL_MANIFEST), ("test", TEST_MANIFEST)]:
    recs = [json.loads(l) for l in open(m) if l.strip()]
    cov = cache.coverage(recs)
    print(f"  {name:5s}: {len(recs):6d} records, coverage {cov * 100:6.2f}%")
    ok &= cov >= 0.98

recs = [json.loads(l) for l in open(TRAIN_MANIFEST) if l.strip()]
v, a, hv, ha = cache.read(recs[0]["clip_id"], 16, 64000, "center")
print(f"\nsample {recs[0]['clip_id']}:")
print(f"  video{tuple(v.shape)} std={v.std():.3f} range=[{v.min():.2f},{v.max():.2f}]")
print(f"  audio{tuple(a.shape)} std={a.std():.4f} has_video={hv} has_audio={ha}")
assert v.std() > 1e-3 and a.std() > 1e-5, "a cached clip decodes to something empty"

shards = sorted(CACHE_OUT.glob("shard_*.bin"))
size = sum(p.stat().st_size for p in shards)
print(f"\n{len(shards)} shards, {size / 1e9:.2f} GB total")

if cache.failures:
    print(f"\nfirst few failures ({len(cache.failures)} total):")
    for cid, why in list(cache.failures.items())[:5]:
        print(f"  {cid}: {why[:120]}")

if ok:
    print("\nREADY. Output tab -> New Dataset -> name it 'davidnet-av-cache' (public),")
    print("then attach it to the training notebook with + Add Input -> Datasets.")
else:
    print("\nINCOMPLETE - commit this notebook again to continue the build.")

## Timing (optional)

In [ ]:
# Optional: how fast does the cache actually serve samples?
#
# This is the number that decides num_workers. If a sample costs ~20 ms, two workers
# saturate two T4s comfortably and there is no reason to pay for more.
import time, random
from src.data.clipcache import ClipCache

cache = ClipCache(CACHE_OUT)
ids = random.Random(0).sample(list(cache.clips), min(200, len(cache)))
t0 = time.time()
for cid in ids:
    cache.read(cid, 16, 64000, "random")
dt = (time.time() - t0) / len(ids)
print(f"{dt * 1000:.1f} ms per sample, single process")
print(f"-> one worker serves ~{1 / dt:.0f} clips/s; a batch of 8 needs {8 * dt:.2f} s of CPU")

## Publish as a Kaggle Dataset

Uploads **only** `av_cache/`, so the cloned repo and the rest of `/kaggle/working` stay
out of it. Needs two more secrets (Add-ons -> Secrets): `KAGGLE_USERNAME` and
`KAGGLE_KEY`, from your account's `kaggle.json`.

If you would rather not add the secrets, skip this cell and use the version's
**Output** tab -> **New Dataset** instead; it works, it just carries the extra files.

In [ ]:
# Publish av_cache as a public Kaggle Dataset the other 9 accounts can attach.
import os, json, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

DATASET_SLUG = "davidnet-av-cache"
DATASET_TITLE = "DAVID-Net AV clip cache (FakeAVCeleb, DVC2)"

_s = UserSecretsClient()
try:
    os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    _have_creds = True
except Exception as e:
    print(f"No Kaggle API secrets ({e}).")
    print("Use the Output tab -> New Dataset instead, or add KAGGLE_USERNAME/KAGGLE_KEY.")
    _have_creds = False

if _have_creds:
    owner = os.environ["KAGGLE_USERNAME"]
    meta = {
        "title": DATASET_TITLE,
        "id": f"{owner}/{DATASET_SLUG}",
        "licenses": [{"name": "unknown"}],
        # The cache is a derived encoding of FakeAVCeleb, so it inherits that corpus's
        # terms. Keep it to collaborators unless you have cleared redistribution.
        "description": (
            "Pre-decoded clip cache for DAVID-Net. Each clip is 24 frames at 224x224 "
            "tiled into one JPEG (quality 87, 4:4:4) plus 6 s of int16 PCM at 16 kHz, "
            "packed into 256 MB shards with an index.json. Derived from FakeAVCeleb "
            "v1.2 and subject to its terms. Built by scripts/build_clip_cache.py."
        ),
    }
    (CACHE_OUT / "dataset-metadata.json").write_text(json.dumps(meta, indent=1))

    size = sum(p.stat().st_size for p in CACHE_OUT.glob("*")) / 1e9
    print(f"uploading {size:.2f} GB from {CACHE_OUT} as {owner}/{DATASET_SLUG} ...")
    print("this takes a while and prints nothing until it finishes")
    r = subprocess.run(["kaggle", "datasets", "create", "-p", str(CACHE_OUT),
                        "-r", "skip", "--dir-mode", "skip"],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    if r.returncode != 0 and "already exists" in (r.stdout + r.stderr):
        print("dataset exists -> pushing a new version instead")
        r = subprocess.run(["kaggle", "datasets", "version", "-p", str(CACHE_OUT),
                            "-m", "rebuild", "-r", "skip", "--dir-mode", "skip"],
                           capture_output=True, text=True)
        print(r.stdout or "", r.stderr or "")
    if r.returncode == 0:
        print(f"\nDone. In the training notebook: + Add Input -> Datasets -> "
              f"{owner}/{DATASET_SLUG}")
        print("Make it public (or share it with your other accounts) so all 10 workers "
              "can attach it.")
    else:
        print(f"\nupload failed (exit {r.returncode}) -- fall back to Output -> New Dataset")